Implement Page Ranking Algorithm

In [1]:
def pagerank(G, alpha=0.85, personalization=None, max_iter=100, tol=1.0e-6, nstart=None, weight='weight', dangling=None):
    if len(G) == 0: 
        return {} 
  
    if not G.is_directed(): 
        D = G.to_directed() 
    else: 
        D = G
        
    # Create a copy in (right) stochastic form 
    W = nx.stochastic_graph(D, weight=weight) 
    N = W.number_of_nodes()
    
    # Choose fixed starting vector if not given 
    if nstart is None: 
        x = dict.fromkeys(W, 1.0 / N) 
    else: 
        # Normalized nstart vector 
        s = float(sum(nstart.values())) 
        x = dict((k, v / s) for k, v in nstart.items()) 
        
    if personalization is None:
        # Assign uniform personalization vector if not given 
        p = dict.fromkeys(W, 1.0 / N)
    else: 
        missing = set(G) - set(personalization) 
        if missing: 
            raise NetworkXError('Personalization dictionary must have a value for every node. Missing nodes %s' % missing) 
        s = float(sum(personalization.values())) 
        p = dict((k, v / s) for k, v in personalization.items())
        
    if dangling is None:
        # Use personalization vector if dangling vector not specified 
        dangling_weights = p 
    else: 
        missing = set(G) - set(dangling) 
        if missing: 
            raise NetworkXError('Dangling node dictionary must have a value for every node. Missing nodes %s' % missing) 
        s = float(sum(dangling.values())) 
        dangling_weights = dict((k, v/s) for k, v in dangling.items())
        
    dangling_nodes = [n for n in W if W.out_degree(n, weight=weight) == 0.0]
    
     # power iteration: make up to max_iter iterations 
    for _ in range(max_iter): 
        xlast = x 
        x = dict.fromkeys(xlast.keys(), 0) 
        danglesum = alpha * sum(xlast[n] for n in dangling_nodes) 
        for n in x:
            # this matrix multiply looks odd because it is 
            # doing a left multiply x^T=xlast^T*W 
            for nbr in W[n]: 
                x[nbr] += alpha * xlast[n] * W[n][nbr][weight] 
            x[n] += danglesum * dangling_weights[n] + (1.0 - alpha) * p[n] 
  
        # check convergence, l1 norm 
        err = sum([abs(x[n] - xlast[n]) for n in x]) 
        if err < N*tol: 
            return x 
    raise NetworkXError('Pagerank: power iteration failed to converge in %d iterations.' % max_iter)

In [2]:
import networkx as nx

In [3]:
G = nx.barabasi_albert_graph(60, 41) 
pr = nx.pagerank(G, 0.4)

In [4]:
print(pr)

{0: 0.028729962168957818, 1: 0.013576960656494622, 2: 0.01295611766110841, 3: 0.01377465554307299, 4: 0.012758394335052593, 5: 0.012567951496638849, 6: 0.013169307176217475, 7: 0.013569860149862934, 8: 0.012571001043582913, 9: 0.012980141838935139, 10: 0.013581235394123939, 11: 0.012759826276774008, 12: 0.012163061914973652, 13: 0.01377465554307299, 14: 0.012552939040619073, 15: 0.013166588199656305, 16: 0.012169042943588747, 17: 0.013169043079998145, 18: 0.012776056337192193, 19: 0.01296072298798823, 20: 0.01316539485169635, 21: 0.012964293584013629, 22: 0.01275665816862181, 23: 0.013165511934448986, 24: 0.013184306648702664, 25: 0.010605945391291293, 26: 0.012971875428554182, 27: 0.01276316854542506, 28: 0.013153812547686779, 29: 0.013371256874462558, 30: 0.01377465554307299, 31: 0.01377465554307299, 32: 0.012969007729284814, 33: 0.01377465554307299, 34: 0.012172188079711597, 35: 0.012565143061063887, 36: 0.012981274554357491, 37: 0.01216813569546561, 38: 0.012956429067117293, 39: 0.

In [5]:
# PageRank Implementation (Concise and Easy)

import numpy as np

# Step 1: Create a simple directed graph
# Example graph: A -> B, A -> C, B -> C, C -> A, D -> C
pages = ['A', 'B', 'C', 'D']
n = len(pages)

links = {
    'A': ['B', 'C'],
    'B': ['C'],
    'C': ['A'],
    'D': ['C']
}

# Step 2: Build adjacency matrix
M = np.zeros((n, n))
for i, p_from in enumerate(pages):
    for j, p_to in enumerate(pages):
        if p_to in links[p_from]:
            M[j][i] = 1 / len(links[p_from])   # distribute PR equally

# Step 3: PageRank calculation
d = 0.85                          # damping factor
PR = np.ones(n) / n               # initial PageRank (equal probability)
iterations = 20                   # number of iterations

for _ in range(iterations):
    PR = (1 - d) / n + d * M.dot(PR)

# Step 4: Print results
print("PageRank Scores:")
for i, p in enumerate(pages):
    print(f"{p}: {PR[i]:.4f}")


PageRank Scores:
A: 0.3725
B: 0.1958
C: 0.3942
D: 0.0375


In [6]:
import networkx as nx

def pagerank(G, alpha=0.85, max_iter=100, tol=1.0e-6):
    if not G.is_directed():
        G = G.to_directed()
    N = len(G)
    pr = dict.fromkeys(G.nodes(), 1.0 / N)
    for _ in range(max_iter):
        prev = pr.copy()
        for n in G:
            pr[n] = (1 - alpha) / N + alpha * sum(prev[m] / len(G[m]) for m in G.predecessors(n))
        if sum(abs(pr[n] - prev[n]) for n in pr) < tol:
            break
    return pr

# Create a sample Barabási–Albert graph
G = nx.barabasi_albert_graph(60, 41)

# Run PageRank manually
pr = pagerank(G, alpha=0.4)

# Display in the same output format
print(pr)

{0: 0.028241108762042173, 1: 0.013374094213700912, 2: 0.01275094529549165, 3: 0.013575134606723362, 4: 0.01335909370461463, 5: 0.01297227545879438, 6: 0.01275328069311012, 7: 0.012967412807594357, 8: 0.011958197580621619, 9: 0.013175158599568583, 10: 0.011794107331334062, 11: 0.012779726545088873, 12: 0.01275094529549165, 13: 0.012771359390725632, 14: 0.013773854619168753, 15: 0.012961250721702655, 16: 0.012347756808683948, 17: 0.012969995434408582, 18: 0.013575134606723362, 19: 0.012158457798641723, 20: 0.011966246834722456, 21: 0.012968435221616299, 22: 0.013161149058008173, 23: 0.012979007898235593, 24: 0.012956628359125234, 25: 0.012969762747020015, 26: 0.01336547339055474, 27: 0.012364752719631066, 28: 0.013571436353893192, 29: 0.01317209924500935, 30: 0.013175384770744925, 31: 0.012949849537585623, 32: 0.01356151196989019, 33: 0.012381704066891323, 34: 0.012963221556940002, 35: 0.013571436353893192, 36: 0.013368147124713438, 37: 0.011763792942603171, 38: 0.013373490997607857, 39:

In [7]:
# Import required library
import networkx as nx

# Create a random graph (Barabási–Albert model)
G = nx.barabasi_albert_graph(60, 41)

# Compute PageRank using NetworkX built-in function
pr = nx.pagerank(G, alpha=0.4)

# Display PageRank values
print("PageRank values for each node:")
print(pr)

PageRank values for each node:
{0: 0.028191876608402847, 1: 0.01294642076998092, 2: 0.01257055556887222, 3: 0.013157281927078098, 4: 0.012967721001638434, 5: 0.013568449342263618, 6: 0.01256684672037416, 7: 0.01176983335693841, 8: 0.012748206210502882, 9: 0.013176538320599375, 10: 0.012756282157293212, 11: 0.013184083244871896, 12: 0.01337836455147496, 13: 0.013368855729500367, 14: 0.01317245562117629, 15: 0.01295931072387783, 16: 0.01196333667305323, 17: 0.012945483348216285, 18: 0.012781266737016126, 19: 0.013775504822815124, 20: 0.012763023620673589, 21: 0.012569921002312998, 22: 0.012755572208581995, 23: 0.012757926857579548, 24: 0.01317477185112518, 25: 0.012756028100217824, 26: 0.013163115004228725, 27: 0.012952624569855851, 28: 0.013582402204454534, 29: 0.011953942921896757, 30: 0.013572774638080197, 31: 0.012564564705841594, 32: 0.01278037813161557, 33: 0.013574039063136475, 34: 0.01296729936314154, 35: 0.012964421611235755, 36: 0.013160281766817196, 37: 0.011783050475319105, 3